# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JunaidAhamed-7777/flyrank-machine-learning/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

Public research page: https://junaidahamed-7777.github.io/flyrank-machine-learning/

This notebook mirrors `docs/index.html` section by section. Run top to bottom to regenerate figures from committed metric receipts (`work/outputs/w06_audit_metrics.json`, `w07_playbook_metrics.json`).


## Abstract

Given a limited weekly content-refresh budget, which pages should an editor refresh first when organic impressions are falling? On a pseudonymized March 2026 panel from the FlyRank internship warehouse (37,385 in-scope pages, decision date 2026-03-31), we ranked pages by decline probability using four post-leakage-fix features and compared a Random Forest to a CTR-headroom rule baseline. Validation combined a single client holdout, leave-one-client-out folds, and a February-to-March time split, with explicit leakage checks that excluded prior-impression transforms tied to the label. The single holdout measured P@50 = 0.38 for the model versus 0.24 for the same-split baseline, while the time-aware check measured 0.44 on the full March frame—both optimistic relative to grouped generalization. The honest generalization read is **LOO-client P@50 = 0.258 ± 0.237** (seven folds at 0.0), a directional decision-support lift over the LOO baseline mean of 0.17, not proof that refresh will recover traffic.

## 1. Question

**Lane:** Refresh prediction reframed as **weekly content-refresh budget allocation**—not a standalone decline classifier.

**Framing question:** *Given a limited weekly content-refresh budget, which pages should a client refresh first to recover the most traffic?*

Editors cannot rewrite every URL. The operational output is a **ranked queue**: scarce writer hours should go to pages where decline signal, exposure, and SERP position suggest the highest leverage. A model that is accurate on average but wrong on ordering still wastes the budget.

**Decision:** Rank in-scope pages by expected traffic-recovery value if refreshed this week (a priority score).

**Action:** Take the top *N* pages and refresh, expand, or apply CTR/snippet fixes as appropriate.

**Cost of errors:** A false positive spends editor time on a page that would not move meaningfully; a false negative leaves demand on the table while impressions erode. This work supports **decision-support** ordering, not causal claims about refresh outcomes.

In [1]:
# Question section — prose only; no code required.

## 2. Data

**Release:** Hugging Face dataset `FlyRank/internship-warehouse` (pseudonymized production search panel).

**Tables used:** `fact_content_daily_performance` (daily GSC aggregates) joined to `dim_content` (content age and metadata).

**Panel window:** Development month **`month=2026-03`** with decision date **2026-03-31**; January–February partitions are read only to build 30- and 60-day impression legs. The analysis frame contains **37,385** page-level rows after filters (positive last-30-day impressions, GSC coverage flag true, content age ≥ 90 days at decision date).

**Excluded by design:** `fact_content_daily_performance_sample` (June 2026 terminal month—natural outcome window for future labels; queried only to confirm existence, never for features or labels). Rows with `gsc_data_available` not true are dropped (pre-tracking zeros are not real zeros). `trend_pct` / `trend_direction` are not used as model inputs (leakage risk; see w03).

**Public safety:** All identifiers are hashed; no client names, live URLs, or private search queries appear in this notebook or the deployed paper.

In [2]:
from pathlib import Path
import json
import shutil

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "work" / "notebooks"
WORK = NOTEBOOK_DIR.parent
REPO_ROOT = WORK.parent
OUT = WORK / "outputs"
FIG = WORK / "figures"
FIG.mkdir(parents=True, exist_ok=True)
DOCS_ASSETS = REPO_ROOT / "docs" / "assets"
DOCS_ASSETS.mkdir(parents=True, exist_ok=True)

audit = json.loads((OUT / "w06_audit_metrics.json").read_text(encoding="utf-8"))
w05 = json.loads((OUT / "w05_model_metrics.json").read_text(encoding="utf-8"))
w07 = json.loads((OUT / "w07_playbook_metrics.json").read_text(encoding="utf-8"))
print("March slice rows:", w07["row_count"])
print("Honest features:", w05["feature_frame_honest"])
print("Leakage demo P@50 with log_imp_prev30:", w05["leakage_demo"]["rf_with_log_imp_prev30_p50"])
print("Leakage demo P@50 honest four features:", w05["leakage_demo"]["rf_honest_four_features_p50"])

March slice rows: 37385
Honest features: ['imp_last30', 'avg_pos_last30', 'ctr_last30', 'content_age_days']
Leakage demo P@50 with log_imp_prev30: 1.0
Leakage demo P@50 honest four features: 0.34


## 3. Methodology

**Label (decline proxy):** `trend_direction = down` when last-30-day impressions fell more than 20% versus the prior 30 days (`imp_prev30 > 0`). Slice base decline rate ≈ **0.21**.

**Honest features (four):** `imp_last30`, `avg_pos_last30`, `ctr_last30`, `content_age_days`. A fifth candidate, `log_imp_prev30`, was **removed** after w05/w06 leakage demos: re-adding it pushed holdout P@50 toward **1.00** because `imp_prev30` enters the label definition.

**Baseline (w04):** CTR-headroom rule—rank by `impressions_90d × max(0, tier_median_ctr − ctr_last30) × tier_weight` (position-tier weights from w04). Reason code `CTR_HEADROOM_EXPOSURE`; action `REFRESH`.

**Model:** Random Forest (`class_weight=balanced_subsample`, `max_depth=10`, `min_samples_leaf=25`, `n_estimators=200`, `random_state=42`) on the four honest features; metrics are **Precision@50** within each evaluation frame (ranking task).

**Validation designs:**
- **Single client holdout (~22/6 clients):** w05 design, reproduced in w06 audit.
- **Leave-one-client-out (LOO):** train on 27 clients, rank within the held-out client's pages; report mean ± std P@50 (headline metric).
- **Time-aware:** train on February decision slice, score all March pages.

**Leakage checks:** w03 `trend_pct` demo; w05 `log_imp_prev30` ablation; w06 re-run with frozen hyperparameters and baseline recomputed on every split.

In [3]:
print("Label definition:", w05["leakage_demo"]["label_definition"])
print("LOO headline:", f"{audit['p50_loo_rf_mean']:.3f} ± {audit['p50_loo_rf_std']:.3f}")

Label definition: (imp_last30 - imp_prev30) / imp_prev30 < -0.20
LOO headline: 0.258 ± 0.237


## 4. Results (vs baseline)

The table below is the w06 audit receipt (Random Forest with four honest features). **Headline metric:** LOO-client P@50 = **0.258 ± 0.237** (seven of 28 client folds at 0.0).

| Regime | P@50 | Notes |
|--------|------|-------|
| In-sample (w05 reference) | 0.92 | Calibration only—not for claims |
| w05 single client-holdout (22/6) | 0.38 | Optimistic vs LOO |
| Leave-one-client-out (mean ± std) | **0.258 ± 0.237** | P@50 within each client |
| Time-aware (train Feb, test Mar) | 0.44 | Not a client-generalization guarantee |
| w04 baseline (full slice) | 0.34 | Reference only |
| w04 baseline (same split as w05 holdout) | 0.24 | Apples-to-apples vs holdout RF |
| w04 baseline (LOO-client mean) | 0.17 | Same LOO protocol as RF |

**Interpretation:** The model's honest performance is LOO-client P@50 = 0.258 ± 0.237—a directional lift over the LOO baseline of 0.17, not the strong lift the single holdout (0.38 vs 0.24) suggested. w05 markdown once cited **0.40**; the executed w05 table and this audit treat that as a historical reference only.

See Figure 1 (model vs baseline by regime) in §7.

In [4]:
import pandas as pd
from IPython.display import display
rows = [
    ("In-sample (w05 reference)", f"{audit['p50_insample_rf']:.2f}", "Calibration only"),
    ("w05 single client-holdout (22/6)", f"{audit['p50_w05_holdout_rf']:.2f}", "Holdout RF"),
    ("Leave-one-client-out (mean ± std)", f"{audit['p50_loo_rf_mean']:.3f} ± {audit['p50_loo_rf_std']:.3f}", "Headline"),
    ("Time-aware (train Feb, test Mar)", f"{audit['p50_time_aware_rf']:.2f}", "Time split"),
    ("w04 baseline (full slice)", f"{audit['global_baseline_p50']:.2f}", "Reference"),
    ("w04 baseline (same split as w05 holdout)", f"{audit['p50_w05_holdout_baseline']:.2f}", "Holdout baseline"),
    ("w04 baseline (LOO-client mean)", f"{audit['loo_baseline_mean']:.3f}", "LOO baseline"),
]
display(pd.DataFrame(rows, columns=["Regime", "P@50", "Notes"]))

,Regime,P@50,Notes
0,In-sample (w05 reference),0.92,Calibration only
1,w05 single client-holdout (22/6),0.38,Holdout RF
2,Leave-one-client-out (mean ± std),0.258 ± 0.237,Headline
3,"Time-aware (train Feb, test Mar)",0.44,Time split
4,w04 baseline (full slice),0.34,Reference
5,w04 baseline (same split as w05 holdout),0.24,Holdout baseline
6,w04 baseline (LOO-client mean),0.173,LOO baseline


## 5. Limitations

1. **Grouped variance:** LOO-client P@50 standard deviation is 0.237 with seven folds at 0.0—queue value is uneven across clients.
2. **Label is observational:** The decline label is GSC momentum, not measured traffic recovery after refresh; no causal refresh experiment was run.
3. **Single development month:** Features and labels are fit on `month=2026-03`; time-aware Feb→Mar is one check, not a full production backtest.
4. **Holdout optimism:** Single client-holdout P@50 (0.38) exceeds LOO mean (0.258); headline claims must use LOO, not holdout.
5. **Base rate context:** Slice decline prevalence ≈ 0.21; P@50 ≈ 0.26 implies ~13 true declines per client-level top-50 batch (~37 false positives)—editorial cost matters.
6. **Rule layer separation:** Archetypes and reason codes describe editorial intent; they are not independently validated outcomes beyond the decline-ranking metric.
7. **Coverage filter:** Excluding incomplete GSC rows changes the population versus raw warehouse counts; findings apply to the filtered contract only.

In [5]:
# Limitations section — prose only.

## 6. Ranked recommendations

The playbook maps each page to an **archetype**, **reason code**, and **action** for human review (w07). The queue sorts by model decline probability; actions are decision-support, not automation.

| Archetype | Signature (summary) | Action |
|-----------|---------------------|--------|
| `HIGH_LEVERAGE_REFRESH` | Page-one/striking tier, CTR below tier median, high exposure (≥ p75) | Refresh content |
| `SNIPPET_OPTIMIZATION` | Top-3 tier, CTR below tier median, high exposure | CTR/snippet fix, not a rewrite |
| `UNDEREXPOSED_DECLINE` | Deep/page_3_5, trend down, below-median exposure | Deprioritise — no refresh this cycle |
| `STABLE_HIGH_PERFORMER` | Top-3 or strong CTR in page-one/striking | Protect — do not touch |
| `CONTENT_AGING` | High exposure, age > 365d, CTR near tier median | Refresh on next cycle |
| `NEW_OR_UNCLEAR` | `no_data` tier or new/flat trend | Manual triage — no model action |
| `LOW_SIGNAL` | `impressions_90d` < 100 | No action |

**Reason codes (fixed vocabulary):** `HIGH_LEVERAGE_REFRESH`, `SNIPPET_CTR_HEADROOM`, `UNDEREXPOSED_DECLINE`, `STABLE_PERFORMER_PROTECT`, `CONTENT_AGING_CYCLE`, `MANUAL_TRIAGE_UNCLEAR`, `BELOW_EXPOSURE_FLOOR`, `INSUFFICIENT_DATA`.

**Human-review rules:** Top 5 pages per `client_hash_id` require editor review before action; `NEW_OR_UNCLEAR` and `LOW_SIGNAL` require manual triage; any page with `impressions_90d` ≥ 50,000 must be reviewed even if ranked below top 50; `reason_code` is mandatory in UI; editor decision overrides the queue.

In [6]:
# Recommendations section — prose only.

## 7. Artifacts the paper embeds

The next cell regenerates:
- `work/figures/model_vs_baseline.svg` (Figure 1)
- `work/figures/archetype_mix.svg` (Figure 2)
- `work/figures/action_distribution.svg` (Figure 3)

Copy committed figures to `docs/assets/` when publishing the HTML paper.

In [ ]:
# w04 CTR-headroom baseline on the w06 Feb->Mar split.
# Fit tier_median_ctr on the February decision slice; score March; P@50 on March.
import os

import numpy as np
import pandas as pd

DECISION_DATE_MAR = "2026-03-31"
DECISION_DATE_FEB = "2026-02-28"
TIER_WEIGHT = {
    "top_3": 1.0,
    "page_1": 0.9,
    "striking": 0.85,
    "page_3_5": 0.7,
    "deep": 0.5,
    "no_data": 0.0,
}


def position_tier(pos: float) -> str:
    if pd.isna(pos) or pos <= 0:
        return "no_data"
    if pos <= 3:
        return "top_3"
    if pos <= 10:
        return "page_1"
    if pos <= 20:
        return "striking"
    if pos <= 50:
        return "page_3_5"
    return "deep"


def precision_at_k(y_true, scores, k: int) -> float:
    frame = pd.DataFrame({"y": list(y_true), "score": list(scores)})
    if frame.empty:
        return 0.0
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0


def load_pages(decision_date: str, month_globs: list[str]) -> pd.DataFrame:
    import duckdb

    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        local = REPO_ROOT / "work" / ".hf_token_local"
        if local.is_file():
            hf_token = local.read_text(encoding="utf-8").strip()
    if not hf_token:
        raise RuntimeError("HF_TOKEN missing")

    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
    rel = "hf://datasets/FlyRank/internship-warehouse"
    dim_content = f"read_parquet('{rel}/dim_content.parquet')"
    paths = ",\n        ".join(
        f"'{rel}/fact_content_daily_performance/{m}/*.parquet'" for m in month_globs
    )
    fact = f"read_parquet([{paths}])"
    sql = f"""
    WITH daily AS (
        SELECT * FROM {fact}
        WHERE report_date <= DATE '{decision_date}'
    ),
    agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date > DATE '{decision_date}' - INTERVAL 90 DAY
                     THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions_90d,
            SUM(CASE WHEN report_date > DATE '{decision_date}' - INTERVAL 30 DAY
                     THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN report_date > DATE '{decision_date}' - INTERVAL 60 DAY
                      AND report_date <= DATE '{decision_date}' - INTERVAL 30 DAY
                     THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN report_date > DATE '{decision_date}' - INTERVAL 30 DAY
                     THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clk_last30,
            AVG(CASE WHEN report_date > DATE '{decision_date}' - INTERVAL 30 DAY
                     THEN gsc_avg_position END) AS avg_pos_last30,
            BOOL_AND(gsc_data_available IS TRUE) AS gsc_ok
        FROM daily
        GROUP BY 1, 2
    ),
    labeled AS (
        SELECT
            a.*,
            CASE
                WHEN a.imp_prev30 = 0 AND a.imp_last30 > 0 THEN 'new'
                WHEN a.imp_prev30 = 0 AND a.imp_last30 = 0 THEN 'flat'
                WHEN a.imp_prev30 > 0 AND (a.imp_last30 - a.imp_prev30) * 100.0 / a.imp_prev30 > 20 THEN 'up'
                WHEN a.imp_prev30 > 0 AND (a.imp_last30 - a.imp_prev30) * 100.0 / a.imp_prev30 < -20 THEN 'down'
                ELSE 'stable'
            END AS trend_direction
        FROM agg a
        WHERE a.imp_last30 > 0 AND a.gsc_ok IS TRUE
    )
    SELECT
        l.client_hash_id AS client_id,
        l.content_hash_id AS content_id,
        l.impressions_90d,
        l.imp_last30,
        l.imp_prev30,
        l.clk_last30,
        l.avg_pos_last30,
        l.trend_direction,
        DATE_DIFF('day', d.content_created_date, DATE '{decision_date}') AS content_age_days
    FROM labeled l
    INNER JOIN {dim_content} d USING (content_hash_id)
    WHERE d.content_created_date <= DATE '{decision_date}'
      AND DATE_DIFF('day', d.content_created_date, DATE '{decision_date}') >= 90
    """
    return con.sql(sql).df()


def enrich(pages: pd.DataFrame) -> pd.DataFrame:
    pages = pages.copy()
    pages["ctr_last30"] = np.where(
        pages["imp_last30"] > 0,
        pages["clk_last30"] / pages["imp_last30"] * 100.0,
        0.0,
    )
    pages["position_tier"] = pages["avg_pos_last30"].map(position_tier)
    pages["is_declining_label"] = pages["trend_direction"].str.lower().eq("down").astype(int)
    tier_median_ctr = pages.groupby("position_tier")["ctr_last30"].transform("median")
    ctr_headroom = (tier_median_ctr - pages["ctr_last30"]).clip(lower=0)
    tier_weight = pages["position_tier"].map(TIER_WEIGHT).fillna(0.0)
    pages["baseline_score"] = pages["impressions_90d"] * ctr_headroom * tier_weight
    return pages


def score_with_train_medians(train: pd.DataFrame, test: pd.DataFrame) -> pd.Series:
    medians = train.groupby("position_tier")["ctr_last30"].median()
    missing = sorted(set(test["position_tier"]) - set(medians.index))
    if missing:
        raise RuntimeError(f"March tiers absent from February medians: {missing}")
    tier_median = test["position_tier"].map(medians)
    headroom = (tier_median - test["ctr_last30"]).clip(lower=0)
    weight = test["position_tier"].map(TIER_WEIGHT).fillna(0.0)
    return test["impressions_90d"] * headroom * weight


print("Loading February slice (w06: 2026-02-28, month=2026-01 and month=2026-02)...")
pages_feb = enrich(load_pages(DECISION_DATE_FEB, ["month=2026-01", "month=2026-02"]))
print("Loading March slice (w06: 2026-03-31, month=2026-01..03)...")
pages_mar = enrich(load_pages(DECISION_DATE_MAR, ["month=2026-01", "month=2026-02", "month=2026-03"]))

time_scores = score_with_train_medians(pages_feb, pages_mar)
p50_time_bl = precision_at_k(pages_mar["is_declining_label"], time_scores, 50)

y = pages_mar["is_declining_label"]
loo_bl = []
for cid in sorted(pages_mar["client_id"].unique()):
    tr = pages_mar["client_id"] != cid
    te = ~tr
    y_tr = y[tr]
    if y_tr.nunique() < 2 or int(te.sum()) == 0:
        continue
    loo_bl.append(precision_at_k(y[te], pages_mar.loc[te, "baseline_score"], 50))
loo_mean = float(np.mean(loo_bl))

stored_loo = float(audit["loo_baseline_mean"])
if abs(loo_mean - stored_loo) >= 1e-9:
    raise RuntimeError(
        f"LOO baseline drifted ({loo_mean} vs stored {stored_loo}); "
        "not writing time_aware_baseline_p50"
    )

audit_path = OUT / "w06_audit_metrics.json"
raw = audit_path.read_text(encoding="utf-8")
rendered = json.dumps(p50_time_bl)
needle = '"time_aware_baseline_p50":'
if needle in raw:
    import re
    raw, n = re.subn(
        r'"time_aware_baseline_p50":\s*[0-9.eE+-]+',
        f'"time_aware_baseline_p50": {rendered}',
        raw,
        count=1,
    )
    if n != 1:
        raise RuntimeError("failed to update time_aware_baseline_p50")
else:
    stripped = raw.rstrip()
    body = stripped[:-1].rstrip()
    raw = body + f',\n  "time_aware_baseline_p50": {rendered}\n}}\n'
audit_path.write_text(raw, encoding="utf-8")
audit["time_aware_baseline_p50"] = json.loads(raw)["time_aware_baseline_p50"]
print(f"feb_rows={len(pages_feb)}")
print(f"mar_rows={len(pages_mar)}")
print(f"time_aware_baseline_p50={audit['time_aware_baseline_p50']}")
print(f"loo_baseline_mean_recomputed={loo_mean:.12f}")
print(f"loo_baseline_mean_stored={stored_loo:.12f}")


In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np

font_path = REPO_ROOT / "docs" / "assets" / "fonts" / "source-serif-4-latin-400-normal.ttf"
font_manager.fontManager.addfont(str(font_path))
plt.rcParams.update({
    "font.family": "Source Serif 4",
    "axes.edgecolor": "#1a1a1a",
    "axes.labelcolor": "#1a1a1a",
    "xtick.color": "#1a1a1a",
    "ytick.color": "#1a1a1a",
    "text.color": "#1a1a1a",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

ACCENT = "#3b4cca"
BASELINE = "#b7b4ac"
base_rate = w07["slice_base_decline_rate"]
regimes = ["LOO-client", "Single holdout\n(22/6)", "Time-aware\n(Feb→Mar)"]
model = [audit["p50_loo_rf_mean"], audit["p50_w05_holdout_rf"], audit["p50_time_aware_rf"]]
baseline = [audit["loo_baseline_mean"], audit["p50_w05_holdout_baseline"], audit["time_aware_baseline_p50"]]
x = np.arange(len(regimes))
w = 0.36
fig, ax = plt.subplots(figsize=(8, 4.4))
ax.bar(x - w / 2, model, w, label="Random Forest (honest features)", color=ACCENT)
ax.bar(x + w / 2, baseline, w, label="w04 CTR-headroom baseline", color=BASELINE)
ax.axhline(base_rate, color="#1a1a1a", linestyle="--", linewidth=1, label=f"Slice base rate ({base_rate:.2f})")
ax.set_ylabel("Precision@50")
ax.set_xticks(x)
ax.set_xticklabels(regimes)
ax.set_ylim(0, 0.55)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
p1 = FIG / "model_vs_baseline.svg"
fig.savefig(p1, transparent=True)
plt.close(fig)

def horizontal(counts, path, xlabel):
    pairs = sorted(counts.items(), key=lambda kv: kv[1])
    labels = [k for k, _ in pairs]
    values = [v for _, v in pairs]
    fig, ax = plt.subplots(figsize=(8, 4.6))
    ax.barh(labels, values, color=ACCENT)
    ax.set_xlabel(xlabel)
    for lab in ax.get_yticklabels():
        lab.set_fontfamily("DejaVu Sans Mono")
        lab.set_fontsize(8)
    fig.tight_layout()
    fig.savefig(path, transparent=True)
    plt.close(fig)

p2 = FIG / "archetype_mix.svg"
p3 = FIG / "action_distribution.svg"
horizontal(w07["archetype_counts"], p2, "Pages")
horizontal(w07["action_counts"], p3, "Pages")

for p in (p1, p2, p3):
    shutil.copy2(p, DOCS_ASSETS / p.name)
    print("Wrote", p, "and copied to", DOCS_ASSETS / p.name)


C:\Users\Junaid\AppData\Local\Temp\ipykernel_23492\1463253548.py:37: UserWarning: Glyph 8594 (\N{RIGHTWARDS ARROW}) missing from font(s) Source Serif 4.
  fig.tight_layout()
C:\Users\Junaid\AppData\Local\Temp\ipykernel_23492\1463253548.py:39: UserWarning: Glyph 8594 (\N{RIGHTWARDS ARROW}) missing from font(s) Source Serif 4.
  fig.savefig(p1, transparent=True)


Wrote C:\aaFiles\aaGithub\FlyrankAI\flyrank-machine-learning\work\figures\model_vs_baseline.svg and copied to C:\aaFiles\aaGithub\FlyrankAI\flyrank-machine-learning\docs\assets\model_vs_baseline.svg
Wrote C:\aaFiles\aaGithub\FlyrankAI\flyrank-machine-learning\work\figures\archetype_mix.svg and copied to C:\aaFiles\aaGithub\FlyrankAI\flyrank-machine-learning\docs\assets\archetype_mix.svg
Wrote C:\aaFiles\aaGithub\FlyrankAI\flyrank-machine-learning\work\figures\action_distribution.svg and copied to C:\aaFiles\aaGithub\FlyrankAI\flyrank-machine-learning\docs\assets\action_distribution.svg


## ML-12 — outreach cuts

**5-minute demo outline:** (1) State the budget-allocation question. (2) Show the data contract slice (37k pages, March 2026). (3) Walk through leakage fix (`log_imp_prev30` removed). (4) Show LOO P@50 = 0.258 ± 0.237 vs baseline 0.17 on the bar chart. (5) Open the archetype table and human-review rules.

**Social post:** When you can only refresh *N* pages this week, ranking beats a flat decline label—but grouped validation matters. On 37k pseudonymized GSC rows, LOO-client P@50 = 0.258 ± 0.237 beat a CTR-headroom baseline (0.17) with high client variance. Chart + paper: https://junaidahamed-7777.github.io/flyrank-machine-learning/

**Employer 3-sentencer:** Built a refresh-prioritisation pipeline on the FlyRank internship warehouse (37k page-level rows, real production panel). Trained an honest Random Forest ranking model, caught label leakage, and audited with leave-one-client-out validation (P@50 = 0.258 ± 0.237). Delivered a human-reviewed action playbook and a public research page with full notebook reproducibility.

## Self-check

- [x] Every section filled — markdown + code for metrics and figures
- [x] Notebook runs top to bottom (metric JSONs + matplotlib)
- [x] No client names, URLs, or private queries
- [x] Claims use observed / measured / directional / decision-support language
- [x] Deployed paper includes Abstract + FlyRank data credit
- [x] Headline metric is LOO-client P@50 = 0.258 ± 0.237 (not w05 0.40)
